# Day 29: LangChain RAG

On Day 11 we built RAG from scratch — embed every doc, compute cosine similarity, assemble the prompt manually.

Today: same pipeline, a fraction of the code. LangChain handles the retrieval.

## Install

In [18]:
%pip install langchain langchain-google-genai langchain-core --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup

In [10]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

def get_text(result):
    content = result["messages"][-1].content
    if isinstance(content, list):
        return " ".join(c.get("text", "") for c in content if isinstance(c, dict))
    return content

## Knowledge Base

In [11]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Our company's refund policy allows returns within 30 days of purchase."),
    Document(page_content="Premium members get free shipping on all orders over $25."),
    Document(page_content="Customer support is available Monday to Friday, 9 AM to 6 PM EST."),
    Document(page_content="Gift cards can be purchased in denominations of $25, $50, and $100."),
    Document(page_content="Orders are typically delivered within 3-5 business days."),
]

print(f"📚 {len(docs)} documents ready")

📚 5 documents ready


## Embed & Index

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(docs)

print("✅ Indexed 5 documents")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Indexed 5 documents


## Retrieval Tool

In [13]:
from langchain.tools import tool

@tool
def retrieve(query: str) -> str:
    """Retrieve relevant company policy documents to answer a question."""
    results = vector_store.similarity_search(query, k=2)
    return "\n".join(doc.page_content for doc in results)

## Create the RAG Agent

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=[retrieve]
)

print("✅ RAG agent ready")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ RAG agent ready


## Test: Questions About Company Policy

In [15]:
result = agent.invoke({"messages": [{"role": "user", "content": "Can I return something I bought 2 weeks ago?"}]})
print("💬", get_text(result))

💬 Yes, you can return something you bought 2 weeks ago as our refund policy allows returns within 30 days of purchase. Our customer support is available Monday to Friday, 9 AM to 6 PM EST if you need further assistance.


In [16]:
result = agent.invoke({"messages": [{"role": "user", "content": "How long does shipping take?"}]})
print("💬", get_text(result))

💬 Orders are typically delivered within 3-5 business days.


In [17]:
# Out of scope — agent should say it doesn't know
result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
print("💬", get_text(result))

💬 I cannot answer this question as my knowledge base is limited to company policy documents.


## Key Takeaways

1. `InMemoryVectorStore` + `GoogleGenerativeAIEmbeddings` replaces **manual embedding loops** from Day 11
2. A `@tool` retriever gives the agent **on-demand access** to your knowledge base
3. The agent retrieves **only when needed** — smarter than always injecting context